# ZnO + CO2 Workflow for Lucas (Geometry -> DFT -> Pathway -> TDDFT)

Run the cells from inside `undergrads/lucas` (recommended), or from repository root.
This notebook is ZnO-only and runs against the simulator workspace in `qml-co2-splitting-mo`.

## Lucas Local Run (Standalone Notebook Use)

One-time setup:

```bash
cd undergrads/lucas
./setup_lucas_env.sh
```

Then run notebook headless:

```bash
source .venv312/bin/activate
cd undergrads/lucas
./run_zno_notebook.sh
```

Or open Jupyter in `undergrads/lucas` and run all cells top-to-bottom.

In [ ]:
from pathlib import Path
import subprocess
import sys
import json
import csv
import math

CWD = Path.cwd()
NOTEBOOK_DIR = next(
    (path for path in [CWD, CWD / "undergrads" / "lucas", CWD.parent / "undergrads" / "lucas"] if (path / "zno-co2-lucas.ipynb").exists()),
    CWD,
)
sim_candidates = [
    CWD,
    CWD / "qml-co2-splitting-mo",
    CWD.parent / "qml-co2-splitting-mo",
    CWD.parent.parent / "qml-co2-splitting-mo",
    NOTEBOOK_DIR.parent.parent / "qml-co2-splitting-mo",
]
LUCAS_DIR = next((path for path in sim_candidates if (path / "run_pipeline.py").exists()), None)
if LUCAS_DIR is None:
    raise FileNotFoundError(
        "Could not resolve simulator workspace 'qml-co2-splitting-mo'. Run from undergrads/lucas or repository root."
    )

print(f"Current directory: {CWD}")
print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Simulator directory: {LUCAS_DIR}")

def run_cmd(args: list[str]) -> None:
    print("$", " ".join(args))
    subprocess.run(args, cwd=LUCAS_DIR, check=True)


## Run Configuration

Edit only this cell if you want to change numerical settings.

In [ ]:
MATERIAL = "ZnO"
SITES = ["top_metal", "top_oxygen", "bridge"]

DFT_KPTS = (3, 3, 1)
DFT_ECUT = 500
DFT_FMAX = 0.05
DFT_STEPS = 180

PATHWAY_FMAX = 0.05
PATHWAY_STEPS = 260

TDDFT_MAX_TRANSITIONS = 50
TDDFT_OSC_THRESHOLD = 1e-3


## 0) Environment Check

This confirms `numpy`, `ase`, and `gpaw` are available in the active environment.

In [ ]:
run_cmd([sys.executable, "-c", "import numpy, ase, gpaw; print('env-ok')"])


## 1) Geometry

Generate ZnO slab and three CO2 adsorption-site starting structures.

In [ ]:
run_cmd([
    sys.executable,
    str(LUCAS_DIR / "build_geometries.py"),
    "--materials",
    MATERIAL,
])


In [ ]:
geometry_dir = LUCAS_DIR / "data" / "geometries" / MATERIAL
expected_files = [
    geometry_dir / "clean_slab.traj",
    geometry_dir / "clean_slab.cif",
    geometry_dir / "co2_top_metal.traj",
    geometry_dir / "co2_top_metal.cif",
    geometry_dir / "co2_top_oxygen.traj",
    geometry_dir / "co2_top_oxygen.cif",
    geometry_dir / "co2_bridge.traj",
    geometry_dir / "co2_bridge.cif",
    geometry_dir / "metadata.json",
]

for path in expected_files:
    state = "OK" if path.exists() else "MISSING"
    print(f"[{state}] {path}")

metadata = json.loads((geometry_dir / "metadata.json").read_text(encoding="utf-8"))
print(json.dumps(metadata, indent=2))


## 2) DFT

Run DFT relaxation for clean ZnO and ZnO+CO2 adsorption cases.
This can take significant compute time.

In [ ]:
dft_cmd = [
    sys.executable,
    str(LUCAS_DIR / "run_dft.py"),
    "--materials",
    MATERIAL,
    "--sites",
    *SITES,
    "--kpts",
    *(str(v) for v in DFT_KPTS),
    "--ecut",
    str(DFT_ECUT),
    "--fmax",
    str(DFT_FMAX),
    "--steps",
    str(DFT_STEPS),
]
run_cmd(dft_cmd)


In [ ]:
def safe_float(value: str | float | int | None) -> float:
    if value is None:
        return float("nan")
    if isinstance(value, (int, float)):
        return float(value)
    text = value.strip()
    if not text:
        return float("nan")
    try:
        return float(text)
    except ValueError:
        return float("nan")

dft_csv = LUCAS_DIR / "results" / "dft_summary.csv"
with dft_csv.open("r", newline="", encoding="utf-8") as handle:
    dft_rows = list(csv.DictReader(handle))

zno_dft = [row for row in dft_rows if row.get("material") == MATERIAL and row.get("site") in SITES]
if not zno_dft:
    raise RuntimeError("No ZnO rows found in dft_summary.csv")

for row in zno_dft:
    print(
        f"{row['site']:>10} | status={row.get('dft_status', '')} "
        f"| Eads={safe_float(row.get('adsorption_energy_ev')):.4f} eV "
        f"| Eg={safe_float(row.get('adsorbed_band_gap_ev')):.4f} eV "
        f"| OCO={safe_float(row.get('oco_angle_deg')):.2f} deg"
    )

print("\nDFT CIF snapshots:")
dft_dir = LUCAS_DIR / "results" / "dft" / MATERIAL
cif_paths = [
    dft_dir / "clean" / "clean_slab_initial.cif",
    dft_dir / "clean" / "clean_slab_optimized.cif",
]
for site in SITES:
    cif_paths.append(dft_dir / site / f"{MATERIAL}_{site}_initial.cif")
    cif_paths.append(dft_dir / site / f"{MATERIAL}_{site}_optimized.cif")
for path in cif_paths:
    state = "OK" if path.exists() else "MISSING"
    print(f"[{state}] {path}")


## 3) Reaction Pathway (CO2 Dissociation Proxy)

Run CO2RR intermediates (`COOH*`, `CO*`, `O*`) and step-wise energetic proxies.

In [ ]:
pathway_cmd = [
    sys.executable,
    str(LUCAS_DIR / "run_co2rr_pathways.py"),
    "--materials",
    MATERIAL,
    "--sites",
    *SITES,
    "--kpts",
    *(str(v) for v in DFT_KPTS),
    "--ecut",
    str(DFT_ECUT),
    "--fmax",
    str(PATHWAY_FMAX),
    "--steps",
    str(PATHWAY_STEPS),
]
run_cmd(pathway_cmd)


In [ ]:
pathway_csv = LUCAS_DIR / "results" / "co2rr_pathway_summary.csv"
with pathway_csv.open("r", newline="", encoding="utf-8") as handle:
    pathway_rows = list(csv.DictReader(handle))

zno_pathway = [row for row in pathway_rows if row.get("material") == MATERIAL and row.get("site") in SITES]
if not zno_pathway:
    raise RuntimeError("No ZnO rows found in co2rr_pathway_summary.csv")

for row in zno_pathway:
    print(
        f"{row['site']:>10} | status={row.get('pathway_status', '')} "
        f"| dG1={safe_float(row.get('deltaG_CO2_to_COOH_ev')):.3f} eV "
        f"| dG2={safe_float(row.get('deltaG_COOH_to_CO_ev')):.3f} eV "
        f"| dG3={safe_float(row.get('deltaG_CO_desorption_ev')):.3f} eV "
        f"| dG4={safe_float(row.get('deltaG_O_removal_ev')):.3f} eV "
        f"| U_lim={safe_float(row.get('limiting_potential_v')):.3f} V"
    )

print("\nPathway CIF snapshots (intermediate initial + optimized):")
pathway_dir = LUCAS_DIR / "results" / "pathways" / MATERIAL
for site in SITES:
    for intermediate in ("cooh", "co", "o"):
        for phase in ("initial", "optimized"):
            cif_path = pathway_dir / site / intermediate / f"{MATERIAL}_{site}_{intermediate}_{phase}.cif"
            state = "OK" if cif_path.exists() else "MISSING"
            print(f"[{state}] {cif_path}")


## 4) TDDFT

Run TDDFT descriptors from the converged DFT structures.

In [ ]:
tddft_cmd = [
    sys.executable,
    str(LUCAS_DIR / "run_tddft.py"),
    "--materials",
    MATERIAL,
    "--sites",
    *SITES,
    "--max-transitions",
    str(TDDFT_MAX_TRANSITIONS),
    "--osc-threshold",
    str(TDDFT_OSC_THRESHOLD),
]
run_cmd(tddft_cmd)


In [ ]:
tddft_csv = LUCAS_DIR / "results" / "tddft_summary.csv"
with tddft_csv.open("r", newline="", encoding="utf-8") as handle:
    tddft_rows = list(csv.DictReader(handle))

zno_tddft = [row for row in tddft_rows if row.get("material") == MATERIAL and row.get("site") in SITES]
if not zno_tddft:
    raise RuntimeError("No ZnO rows found in tddft_summary.csv")

for row in zno_tddft:
    print(
        f"{row['site']:>10} | status={row.get('tddft_status', '')} "
        f"| onset={safe_float(row.get('tddft_onset_ev')):.4f} eV "
        f"| peak={safe_float(row.get('tddft_peak_energy_ev')):.4f} eV "
        f"| total_osc={safe_float(row.get('tddft_total_oscillator_strength')):.4f}"
    )


## Solution Cell (ZnO Site Ranking from DFT + TDDFT)

This computes the same style of screening score used in the project analysis scripts and writes a ZnO-only ranked table.

In [ ]:
TARGET_ADSORPTION_ENERGY_EV = -0.70
TARGET_BAND_GAP_EV = 2.40
TARGET_ONSET_EV = 2.20

def gaussian_score(value: float, target: float, sigma: float) -> float:
    if not math.isfinite(value):
        return 0.0
    return math.exp(-((value - target) ** 2) / (2.0 * sigma**2))

def geometry_activation_score(avg_bond: float, oco_angle: float) -> float:
    if not math.isfinite(avg_bond) or not math.isfinite(oco_angle):
        return 0.0
    bond_stretch = max(0.0, avg_bond - 1.16) / 0.20
    angle_bend = max(0.0, 180.0 - oco_angle) / 40.0
    return min(1.0, 0.5 * bond_stretch + 0.5 * angle_bend)

tddft_by_site = {row["site"]: row for row in zno_tddft}
ranked: list[dict[str, float | str]] = []

for dft in zno_dft:
    site = dft["site"]
    tddft = tddft_by_site.get(site, {})

    adsorption_energy = safe_float(dft.get("adsorption_energy_ev"))
    band_gap = safe_float(dft.get("adsorbed_band_gap_ev"))
    onset = safe_float(tddft.get("tddft_onset_ev"))
    co1 = safe_float(dft.get("co_bond_1_ang"))
    co2 = safe_float(dft.get("co_bond_2_ang"))
    oco_angle = safe_float(dft.get("oco_angle_deg"))
    avg_bond = (co1 + co2) / 2.0

    adsorption_score = gaussian_score(adsorption_energy, TARGET_ADSORPTION_ENERGY_EV, 0.50)
    band_gap_score = gaussian_score(band_gap, TARGET_BAND_GAP_EV, 0.70)
    onset_score = gaussian_score(onset, TARGET_ONSET_EV, 0.80)
    activation_score = geometry_activation_score(avg_bond, oco_angle)

    total_score = (
        0.35 * adsorption_score
        + 0.25 * band_gap_score
        + 0.25 * onset_score
        + 0.15 * activation_score
    )

    ranked.append(
        {
            "site": site,
            "total_score": total_score,
            "adsorption_energy_ev": adsorption_energy,
            "adsorbed_band_gap_ev": band_gap,
            "tddft_onset_ev": onset,
            "oco_angle_deg": oco_angle,
        }
    )

ranked.sort(key=lambda row: float(row["total_score"]), reverse=True)

print("ZnO site ranking (higher score is better):")
for idx, row in enumerate(ranked, start=1):
    print(
        f"{idx}. {row['site']:<10} | score={float(row['total_score']):.4f} "
        f"| Eads={float(row['adsorption_energy_ev']):.4f} eV "
        f"| Eg={float(row['adsorbed_band_gap_ev']):.4f} eV "
        f"| onset={float(row['tddft_onset_ev']):.4f} eV "
        f"| OCO={float(row['oco_angle_deg']):.2f} deg"
    )

solution_csv = LUCAS_DIR / "results" / "zno_solution_ranked.csv"
with solution_csv.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(ranked[0].keys()))
    writer.writeheader()
    writer.writerows(ranked)

print(f"Wrote: {solution_csv}")
